

# 📝 1️⃣ Introduction

## 📌 Project Introduction

The rapid growth of online and competitive gaming has significantly influenced modern lifestyles, especially among adolescents and young adults. While gaming can be a source of entertainment and cognitive stimulation, excessive gaming has been associated with sleep disruption, reduced social interaction, and potential behavioral addiction.

This project aims to analyze gaming behavior patterns and build a machine learning classification model to predict **Gaming Addiction Risk Level** based on demographic, behavioral, and lifestyle-related features.

The notebook demonstrates a complete end-to-end machine learning workflow, including:

* Data preprocessing
* Outlier removal
* Encoding categorical features
* Feature scaling
* Model training (5 classification models)
* Model evaluation using confusion matrix, classification report, and ROC-AUC curve
* Feature importance analysis

This project can serve as:

* A practical ML learning project
* A behavioral analytics study
* A resume-level end-to-end ML portfolio project

---

# 📑 2️⃣ Table of Contents

## 📚 Table of Contents

1. 🔍 Problem Statement
2. 📂 Dataset Overview
3. 🧹 Data Cleaning & Preprocessing

   * Missing Value Handling
   * Outlier Removal (IQR Method)
   * Label Encoding
   * Feature Scaling (MinMaxScaler)
4. 📊 Exploratory Data Analysis (EDA)

   * Univariate Analysis
   * Bivariate Analysis
   * Correlation Heatmap
5. ✂ Train-Test Split
6. 🤖 Model Building

   * Logistic Regression
   * Random Forest
   * Support Vector Machine (SVM)
   * K-Nearest Neighbors (KNN)
   * Decision Tree
7. 📈 Model Evaluation

   * Confusion Matrix
   * Classification Report
   * ROC-AUC Curve
8. 🌳 Feature Importance Analysis
9. 📊 Model Comparison
10. 🧠 Conclusion & Key Insights

---

# 📊 3️⃣ Executive Summary

## 📌 Executive Summary

This project explores the relationship between gaming habits, sleep behavior, and addiction risk levels using a structured behavioral dataset.

After performing data preprocessing and cleaning (including outlier removal and scaling), five classification models were trained to predict **Gaming Addiction Risk Level**:

* Logistic Regression
* Random Forest
* Support Vector Machine
* K-Nearest Neighbors
* Decision Tree

Model performance was evaluated using:

* Confusion Matrix
* Precision, Recall, F1-Score
* ROC-AUC Curve

The results indicate that behavioral factors such as daily gaming hours, sleep duration, social isolation score, and monthly spending significantly influence addiction risk classification.

This notebook demonstrates a full machine learning lifecycle suitable for academic, research, and professional portfolio use.

---




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/shaistashahid/gaming-and-mental-health/Gaming and Mental Health.csv")
print(df.shape)
print(df.info())
print(df.describe())
print(df.head())

In [ ]:
df = df.dropna()  


In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.histplot(df["daily_gaming_hours"], kde=True)
plt.title("Daily Gaming Hours Distribution")
plt.show()


In [ ]:
plt.figure()
sns.countplot(x="sleep_quality", data=df)
plt.title("Sleep Quality Distribution")
plt.show()


In [ ]:
plt.figure()
sns.countplot(x="gaming_addiction_risk_level", data=df)
plt.title("Addiction Risk Level")
plt.show()


In [ ]:
plt.figure()
sns.scatterplot(x="daily_gaming_hours", y="sleep_hours", data=df)
plt.title("Gaming Hours vs Sleep Hours")
plt.show()


In [ ]:
plt.figure()
sns.boxplot(x="sleep_quality", y="daily_gaming_hours", data=df)
plt.title("Gaming Hours by Sleep Quality")
plt.show()


In [ ]:
plt.figure()
sns.boxplot(x="gaming_addiction_risk_level", y="daily_gaming_hours", data=df)
plt.title("Gaming Hours by Addiction Risk")
plt.show()


In [ ]:
plt.figure()
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
plt.figure()
sns.scatterplot(x="daily_gaming_hours", y="social_isolation_score", data=df)
plt.show()


In [ ]:
plt.figure()
sns.boxplot(x="gaming_addiction_risk_level", y="exercise_hours_weekly", data=df)
plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns


In [ ]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]


In [ ]:
categorical_cols = df.select_dtypes(include='object').columns


In [ ]:
le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])


In [ ]:
X = df.drop("gaming_addiction_risk_level", axis=1)
y = df["gaming_addiction_risk_level"]


In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier()
}


In [ ]:
for name, model in models.items():
    print("=========", name, "=========")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nROC-AUC Score:")
    y_prob = model.predict_proba(X_test)
    print(roc_auc_score(y_test, y_prob, multi_class="ovr"))
    
    print("\n\n")


In [ ]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)

for i in range(len(np.unique(y))):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

importance = model.feature_importances_

feature_importance = pd.Series(importance, index=X.columns)
feature_importance.sort_values(ascending=False).plot(kind="bar")
plt.title("Feature Importance")
plt.show()
